In [1]:
# ==========================================
# Cell 1: Imports and Directory Setup
# ==========================================
import os
import glob
import pickle
from astropy.io import fits
from astropy.cosmology import FlatLambdaCDM
from astropy.units import Quantity
from astropy.table import Table
from tqdm import tqdm

import slsim.Sources as sources
from utils import extract_lens_properties, make_multiband_images_and_rgb_image, plot_montage

import warnings
warnings.filterwarnings("ignore")

%load_ext autoreload
%autoreload 2

# Define and create output directories
img_dir = "../data/lenses/images/"
table_dir = "../data/lenses/"
os.makedirs(img_dir, exist_ok=True)
os.makedirs(table_dir, exist_ok=True)

In [2]:
# ==========================================
# Cell 2: Load Lenses and Inject Field Galaxies
# ==========================================
lensed_quasars_all = []
lens_files = glob.glob("../saved_objects/lensed_quasars_*.pkl")

print(f"Found {len(lens_files)} lens files. Loading...")
for file in lens_files:
    with open(file, "rb") as f:
        lensed_quasars_all.extend(pickle.load(f))
        
print(f"Total lenses loaded: {len(lensed_quasars_all)}")

Found 1 lens files. Loading...
Total lenses loaded: 50000


In [3]:
# ==========================================
# Cell 3: Extract Properties and Save Table
# ==========================================
print("Extracting physical properties into catalog table...")
table_lensed_quasars = extract_lens_properties(lensed_quasars_all, all_bands=['g', 'r', 'i', 'z', 'y'], max_num_images=5)

table_path = os.path.join(table_dir, f"table_lensed_quasars_{len(table_lensed_quasars)}.fits")
table_lensed_quasars.write(table_path, format="fits", overwrite=True)
print(f"Saved catalog to {table_path}")

Extracting physical properties into catalog table...


Extracting lens properties:   0%|          | 0/50000 [00:00<?, ?it/s]

Extracting lens properties: 100%|██████████| 50000/50000 [04:35<00:00, 181.71it/s]


Saved catalog to ../data/lenses/table_lensed_quasars_50000.fits


In [5]:
# ==========================================
# Cell 4: Setup Field Galaxy Population
# ==========================================
cosmo = FlatLambdaCDM(H0=70, Om0=0.3)
sky_area_galaxy = Quantity(2, "deg2")

print("Loading SkyPy catalog for field galaxies...")
all_galaxy_catalog = Table.read(f'../catalogs/skypy_all_galaxies_{sky_area_galaxy.value}deg2.fits', format='fits')

field_galaxy_pop = sources.Galaxies(
    galaxy_list=all_galaxy_catalog, 
    kwargs_cut={"band": "i", "band_max": 26, "z_min": 0.01, "z_max": 5.0}, 
    cosmo=cosmo, 
    sky_area=sky_area_galaxy, 
    catalog_type="skypy"
)

Loading SkyPy catalog for field galaxies...


In [ ]:
print("Injecting field galaxies into lens cutouts...")
for lens in lensed_quasars_all:
    lens.add_field_galaxies(field_galaxies=field_galaxy_pop.draw_galaxies(area=Quantity(70, "arcsec2")))

Injecting field galaxies into lens cutouts...


In [ ]:
# ==========================================
# Cell 5: Render and Save FITS Images
# ==========================================
print("Rendering images (coadd_years=1) and saving to disk. This will take time...")

for obj, row in tqdm(zip(lensed_quasars_all, table_lensed_quasars), total=len(table_lensed_quasars), desc="Processing lenses"):
    obj_id = row["Lens ID"]
    
    # Generate 1-year co-add images
    multiband_image, _ = make_multiband_images_and_rgb_image(
        lens_class=obj,
        bands=['g', 'r', 'i', 'z', 'y'],
        num_pix=41,
        coadd_years=1,  # 1-year co-add images
        add_noise=True,
        rgb_bands=['i', 'r', 'g'],
        rgb_stretch=0.5,
    )

    for band in ['g', 'r', 'i', 'z', 'y']:
        fits_path = os.path.join(img_dir, f"{obj_id}_{band}.fits")
        fits.writeto(fits_path, multiband_image[band], overwrite=True)

print("Lens rendering and extraction complete!")